In [ ]:
import os
import pickle

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
import re
import base64

from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import pandas as pd
import numpy as np
import gspread

from oauth2client.service_account import ServiceAccountCredentials

import tkinter as tk

from tkinter import (
    filedialog,
    messagebox,
    ttk
)

# =========================================================
# GOOGLE AUTH
# =========================================================

scope = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive"
]

creds = ServiceAccountCredentials.from_json_keyfile_name(
    "service_account.json",
    scope
)

client = gspread.authorize(creds)

# =========================================================
# SHEETS
# =========================================================

FLOW_ANALYSIS_SHEET = "Final_Corrected_Flow_Analysis_Sheet"

FLOW_MAP_FILE = "Flow_Map_File"

# =========================================================
# HELPERS
# =========================================================

def load_file(file_path):

    if file_path.lower().endswith(".csv"):

        encodings = [
            "utf-8",
            "utf-8-sig",
            "latin1",
            "cp1252"
        ]

        for enc in encodings:

            try:

                return pd.read_csv(
                    file_path,
                    encoding=enc
                )

            except:
                pass

        raise Exception(
            f"Unable to read CSV file: {file_path}"
        )

    else:

        return pd.read_excel(file_path)


def clean_text(x):

    if pd.isna(x):
        return ""

    return str(x).strip().lower()


# ---------------------------------------------------------

def clean_channel(x):

    x = clean_text(x)

    whatsapp_vals = [
        "wa",
        "whatsapp",
        "whats app",
        "whats_app",
        "whats-app",
        "watsapp",
        "whatsup",
        "whatsap"
    ]

    email_vals = [
        "email",
        "mail",
        "e-mail"
    ]

    if x in whatsapp_vals:
        return "whatsapp"

    if x in email_vals:
        return "email"

    return x

# ---------------------------------------------------------

def to_num(x):

    try:

        if pd.isna(x):
            return 0

        x = str(x)

        x = x.replace(",", "")
        x = x.replace("%", "")
        x = x.strip()

        return float(x)

    except:
        return 0


# ---------------------------------------------------------

def r2(x):

    try:

        return round(float(x), 1)

    except:

        return 0


# =========================================================
# MAIN FUNCTION
# =========================================================

def get_meta_alerts(selected_brand):

    meta_alerts = {}

    try:

        SCOPES = [
            "https://www.googleapis.com/auth/gmail.readonly"
        ]

        creds = None

        if os.path.exists("token.pickle"):

            with open(
                "token.pickle",
                "rb"
            ) as token:

                creds = pickle.load(token)

        if not creds or not creds.valid:

            if (
                creds
                and creds.expired
                and creds.refresh_token
            ):

                creds.refresh(Request())

            else:

                flow = InstalledAppFlow.from_client_secrets_file(
                    "gmail_automate.json",
                    SCOPES
                )

                creds = flow.run_local_server(
                    port=0
                )

            with open(
                "token.pickle",
                "wb"
            ) as token:

                pickle.dump(
                    creds,
                    token
                )

        service = build(
            "gmail",
            "v1",
            credentials=creds
        )

        results = service.users().messages().list(

            userId="me",

            q='subject:"KwikEngage- Template Category Updates" newer_than:30d',

            maxResults=100

        ).execute()

        messages = results.get(
            "messages",
            []
        )

        print(
            "META ALERT MAILS:",
            len(messages)
        )

        for m in messages:

            try:

                msg = service.users().messages().get(
                    userId="me",
                    id=m["id"]
                ).execute()

                headers = msg["payload"]["headers"]

                subject = ""

                for h in headers:

                    if h["name"] == "Subject":

                        subject = h["value"]

                        break

                print("SUBJECT:", subject)

                payload = msg["payload"]
                print("PAYLOAD KEYS:", payload.keys())

                data = ""

                if "data" in payload.get("body", {}):

                    data = payload["body"]["data"]

                elif "parts" in payload:

                    for part in payload["parts"]:

                       if "data" in part.get("body", {}):

                                data = part["body"]["data"]

                                break

                if not data:

                    continue

                text = base64.urlsafe_b64decode(

                    data
                ).decode(
                    "utf-8",
                    errors="ignore"
                )


                
                brand_match = re.search(
                    r"Hi\s+(.*?)\s*\(",
                    text,
                    flags=re.IGNORECASE
                )

                if not brand_match:

                    continue

                mail_brand = brand_match.group(1).strip().lower()

                if "neuro" in mail_brand:

                    detected_brand = "Neurogum"

                elif "durex" in mail_brand:

                    detected_brand = "Durex"

                elif "avon" in mail_brand:

                    detected_brand = "Avon"

                elif "enamor" in mail_brand:

                    detected_brand = "Enamor"

                else:

                    continue

                if detected_brand != selected_brand:

                    continue

                templates = re.findall(
                    r"Template\s*Name.*?([A-Za-z0-9_\-]+)",
                    text,
                    flags=re.IGNORECASE | re.DOTALL
                    
                )

                print("RAW MATCH:", templates)

                print("TEMPLATES FOUND:", templates)

                for template in templates:

                    template = template.strip().lower()

                    meta_alerts[template] = (
                        f"Review Template: {template} "
                        f"(META category updated in last 30 days)"
                    )

                    print(
                        "META ALERT ADDED:",
                        template
                    )

            except Exception as e:

                print(
                    "MAIL ERROR:",
                    str(e)
                )

    except Exception as e:

        print(
            "META ALERT ERROR:",
            str(e)
        )

    print(
        "FINAL META ALERTS:",
        meta_alerts
    )

    return meta_alerts

def run_pipeline(

    brand,
    kwik_file,
    orders_file,
    session_file,
    start_date,
    end_date
):

    print("=" * 70)
    print("RUNNING FOR:", brand)
    print("=" * 70)

    meta_alerts = get_meta_alerts(brand)

    print(
        "META ALERTS FOUND:",
        len(meta_alerts)
    )

    # =====================================================
    # DATE RANGE
    # =====================================================

    start_date = pd.to_datetime(start_date)

    end_date = pd.to_datetime(end_date)

    duration = (
        f"{start_date.strftime('%d-%b')} "
        f"to "
        f"{end_date.strftime('%d-%b')}"
    )

    # =====================================================
    # OPEN OUTPUT SHEET
    # =====================================================

    output_book = client.open(FLOW_ANALYSIS_SHEET)

    ws = output_book.worksheet(brand)

    print("OUTPUT SHEET OPENED")

    # =====================================================
    # OPEN FLOW MAP
    # =====================================================

    flow_book = client.open(FLOW_MAP_FILE)

    flow_ws = flow_book.worksheet(brand)

    flow_df = pd.DataFrame(
        flow_ws.get_all_records()
    )

    flow_df.columns = [
        str(c).strip()
        for c in flow_df.columns
    ]

    print("FLOW MAP LOADED")

    # =====================================================
    # CLEAN FLOW MAP
    # =====================================================

    flow_df["Flows"] = (

        flow_df["Flows"]

        .replace("", np.nan)

        .ffill()

        .astype(str)

        .str.strip()
    )

    flow_df["Flows UTM"] = (

        flow_df["Flows UTM"]

        .astype(str)

        .apply(clean_text)
    )

    # =====================================================
    # CLEAN TYPE COLUMN
    # =====================================================

    flow_df["Type"] = (

        flow_df["Type"]

        .astype(str)

        .str.strip()

        .str.upper()
    )

    # =====================================================
    # LOAD KWIK FILE
    # =====================================================

    kwik_df = load_file(kwik_file)

    kwik_df.columns = [
        str(c).strip()
        for c in kwik_df.columns
    ]

    print("KWIK FILE LOADED")

    # =====================================================
    # CLEAN KWIK FILE
    # =====================================================

    kwik_df["Name"] = (
        kwik_df["Name"]
        .astype(str)
        .str.strip()
    )

    kwik_df["Channel"] = (
        kwik_df["Channel"]
        .apply(clean_channel)
    )

    metric_cols = [
        "Sent",
        "Delivered",
        "Seen",
        "Clicks"
    ]

    for c in metric_cols:

        kwik_df[c] = kwik_df[c].apply(to_num)

    # =====================================================
    # ENGAGEMENT METRICS
    # =====================================================

    kwik_df["Delivery %"] = np.where(

        kwik_df["Sent"] > 0,

        (
            kwik_df["Delivered"]
            /
            kwik_df["Sent"]
        ) * 100,

        0
    )

    kwik_df["Open rate %"] = np.where(

        kwik_df["Delivered"] > 0,

        (
            kwik_df["Seen"]
            /
            kwik_df["Delivered"]
        ) * 100,

        0
    )

    kwik_df["CTR %"] = np.where(

        kwik_df["Delivered"] > 0,

        (
            kwik_df["Clicks"]
            /
            kwik_df["Delivered"]
        ) * 100,

        0
    )

    # =====================================================
    # SESSION FILE
    # =====================================================

    session_df = load_file(session_file)

    session_df.columns = [
        str(c).strip()
        for c in session_df.columns
    ]

    print("SESSION FILE LOADED")

    # =====================================================
    # SESSION COLUMNS
    # =====================================================

    session_day_col = None
    session_utm_col = None
    session_count_col = None

    for c in session_df.columns:

        low = c.lower().strip()

        if low == "day":
            session_day_col = c

        elif low == "utm campaign":
            session_utm_col = c

        elif low == "sessions":
            session_count_col = c


    

   


    print("SESSION DAY:", session_day_col)
    print("SESSION UTM:", session_utm_col)
    print("SESSION COUNT:", session_count_col)

    # =====================================================
    # CLEAN SESSION FILE
    # =====================================================

    session_df[session_day_col] = pd.to_datetime(

        session_df[session_day_col],

        errors="coerce",

        dayfirst=True

    ).dt.normalize()

    session_df[session_utm_col] = (

        session_df[session_utm_col]

        .astype(str)

        .apply(clean_text)
    )

    session_df[session_count_col] = (

        session_df[session_count_col]

        .apply(to_num)
    )

    # =====================================================
    # FILTER SESSION DATE
    # =====================================================

    session_df = session_df[

        (
            session_df[session_day_col]
            >=
            start_date
        )

        &

        (
            session_df[session_day_col]
            <=
            end_date
        )
    ]

    print("FILTERED SESSION ROWS:", len(session_df))

    # =====================================================
    # LOAD ORDER FILE
    # =====================================================

    order_df = load_file(orders_file)

    order_df.columns = [
        str(c).strip()
        for c in order_df.columns
    ]

    print("ORDER FILE LOADED")

    # =====================================================
    # ORDER COLUMNS
    # =====================================================

    created_col = None
    utm_col = None
    medium_col = None
    revenue_col = None

    for c in order_df.columns:

        low = c.lower()

        if "created at" in low:
            created_col = c

        if "utm campaign" in low:
            utm_col = c

        if "utm medium" in low:
            medium_col = c

        if "grand total" in low:
            revenue_col = c

    print("CREATED COL:", created_col)
    print("UTM COL:", utm_col)
    print("MEDIUM COL:", medium_col)
    print("REVENUE COL:", revenue_col)

    # =====================================================
    # CLEAN ORDER FILE
    # =====================================================

    order_df[created_col] = pd.to_datetime(

        order_df[created_col],

        errors="coerce",

        dayfirst=True

    ).dt.normalize()

    order_df[utm_col] = (

        order_df[utm_col]

        .astype(str)

        .apply(clean_text)
    )

    order_df[medium_col] = (

        order_df[medium_col]

        .apply(clean_channel)
    )

    order_df[revenue_col] = (

        order_df[revenue_col]

        .apply(to_num)
    )
    # =====================================================
    # FILTER ORDER DATE
    # =====================================================

    order_df = order_df[

        (
            order_df[created_col]
            >=
            start_date
        )

        &

        (
            order_df[created_col]
            <=
            end_date
        )
    ]

    print("FILTERED ORDER ROWS:", len(order_df))

    # =====================================================
    # FINAL ROWS
    # =====================================================

    final_rows = []

    unique_flows = (

        flow_df["Flows"]

        .dropna()

        .unique()
    )

    # =====================================================
    # FLOW LOOP
    # =====================================================

    for flow in unique_flows:

        print("=" * 60)
        print("FLOW:", flow)

        utm_rows = flow_df[

            flow_df["Flows"] == flow

        ]["Flows UTM"]

        utms = []

        for u in utm_rows:

            u = clean_text(u)

            if u != "":
                utms.append(u)

        utms = list(set(utms))

        print("UTMS:", utms)

        # =================================================
        # FLOW TYPE
        # =================================================

        flow_types = flow_df[
            flow_df["Flows"] == flow
        ]["Type"].dropna().unique()

        flow_templates = flow_df[
            flow_df["Flows"] == flow
        ]["Templates"]

        flow_templates = [

            str(x).strip().lower()

            for x in flow_templates

            if str(x).strip()
        ]

        flow_type = "MARKETING"

        if len(flow_types) > 0:

            flow_type = flow_types[0]

        print("FLOW TYPE:", flow_type)

        # =================================================
        # FLOW DATA
        # =================================================

        current_flow_df = kwik_df[

            kwik_df["Name"]

            .str.lower()

            .str.strip()

            ==

            str(flow).lower().strip()
        ]

        print("FLOW DF LEN:", len(current_flow_df))

        if len(current_flow_df) == 0:
            continue

        # =================================================
        # CHANNEL LOOP
        # =================================================

        channels = (

            current_flow_df["Channel"]

            .dropna()

            .unique()
        )

        print("CHANNELS:", channels)

        for channel in channels:

            print("CHANNEL:", channel)

            ch_df = current_flow_df[

                current_flow_df["Channel"]
                ==
                channel
            ]

            # =============================================
            # ENGAGEMENT
            # =============================================

            sent = ch_df["Sent"].sum()

            delivered = ch_df["Delivered"].sum()

            if flow_type == "UTILITY":

                spends = delivered * 0.25

            else:

                spends = delivered * 0.9

            opens = ch_df["Seen"].sum()

            clicks = ch_df["Clicks"].sum()

            delivery_rate = (
                (delivered / sent) * 100
                if sent > 0 else 0
            )

            open_rate = (
                (opens / delivered) * 100
                if delivered > 0 else 0
            )

            ctr = (
                (clicks / delivered) * 100
                if delivered > 0 else 0
            )

            # =============================================
            # SESSIONS
            # =============================================

            filtered_sessions = session_df[

                session_df[session_utm_col]

                .isin(utms)
            ]

            sessions = (

                filtered_sessions[session_count_col]

                .sum()
            )

            print("SESSIONS:", sessions)

            # =============================================
            # ORDERS + REVENUE
            # =============================================

            filtered_orders = order_df[

                (
                    order_df[utm_col]

                    .isin(utms)
                )

                &

                (
                    order_df[medium_col]
                    ==
                    channel
                )
            ]

            print("MATCHED ORDERS ROWS:", len(filtered_orders))

            orders = len(filtered_orders)

            revenue = (

                filtered_orders[revenue_col]

                .sum()
            )

            print("ORDERS:", orders)
            print("REVENUE:", revenue)

            # =============================================
            # KPI
            # =============================================

            cvr = (
                (orders / clicks) * 100
                if clicks > 0 else 0
            )

            aov = (
                revenue / orders
                if orders > 0 else 0
            )

            roas = (
                revenue / spends
                if spends > 0 else 0
            )

            # =============================================
            # FINAL ROW
            # =============================================

            meta_warning = ""

            print("FLOW TEMPLATES:", flow_templates)
            print("META ALERT KEYS:", list(meta_alerts.keys()))
            for template in flow_templates:

                if template in meta_alerts:

                    meta_warning = meta_alerts[template]

                    break

            row = [

                duration,
                flow,
                channel,

                r2(sent),
                r2(delivered),
                r2(delivery_rate),
                r2(spends),
                r2(opens),
                r2(open_rate),
                r2(clicks),
                r2(ctr),
                r2(sessions),
                r2(orders),
                r2(cvr),
                r2(revenue),
                r2(aov),
                r2(roas),
                meta_warning
            ]

            final_rows.append(row)

            print("ROW APPENDED")

    # =====================================================
    # FINAL DEBUG
    # =====================================================

    print("=" * 70)
    print("FINAL ROWS LENGTH:", len(final_rows))

    if len(final_rows) > 0:

        print("FIRST ROW SAMPLE:")
        print(final_rows[0])

    print("=" * 70)

    # =====================================================
    # PUSH TO SHEET
    # =====================================================

    if len(final_rows) > 0:

        try:

            ws.append_rows(

                final_rows,

                value_input_option="USER_ENTERED"
            )

            print("DATA PUSHED SUCCESSFULLY")

        except Exception as e:

            print("GOOGLE SHEET PUSH ERROR")
            print(str(e))

    else:

        print("NO DATA FOUND")

    print("=" * 70)
    print("DONE")
    print("=" * 70)
# =========================================================
# TKINTER UI
# =========================================================

root = tk.Tk()

root.title("Flow Automation")

root.geometry("500x700")

# =========================================================
# BRAND
# =========================================================

tk.Label(root, text="Brand").pack()

brand_var = tk.StringVar()

brand_dropdown = ttk.Combobox(

    root,

    textvariable=brand_var,

    values=[
        "Durex",
        "Avon",
        "Enamor",
        "Neurogum"
    ],

    state="readonly"
)

brand_dropdown.pack()

# =========================================================
# FILE PICKER
# =========================================================

def browse(entry):

    file = filedialog.askopenfilename()

    entry.delete(0, tk.END)

    entry.insert(0, file)

# =========================================================
# KWIK FILE
# =========================================================

tk.Label(root, text="KwikEngage File").pack()

kwik_entry = tk.Entry(root, width=60)

kwik_entry.pack()

tk.Button(
    root,
    text="Browse",
    command=lambda: browse(kwik_entry)
).pack()

# =========================================================
# ORDERS FILE
# =========================================================

tk.Label(root, text="Orders File").pack()

orders_entry = tk.Entry(root, width=60)

orders_entry.pack()

tk.Button(
    root,
    text="Browse",
    command=lambda: browse(orders_entry)
).pack()

# =========================================================
# SESSION FILE
# =========================================================

tk.Label(root, text="Session File").pack()

session_entry = tk.Entry(root, width=60)

session_entry.pack()

tk.Button(
    root,
    text="Browse",
    command=lambda: browse(session_entry)
).pack()

# =========================================================
# START DATE
# =========================================================

tk.Label(root, text="Start Date (YYYY-MM-DD)").pack()

start_entry = tk.Entry(root)

start_entry.pack()

# =========================================================
# END DATE
# =========================================================

tk.Label(root, text="End Date (YYYY-MM-DD)").pack()

end_entry = tk.Entry(root)

end_entry.pack()

# =========================================================
# RUN BUTTON
# =========================================================

def run():

    try:

        run_pipeline(

            brand=brand_var.get(),

            kwik_file=kwik_entry.get(),

            orders_file=orders_entry.get(),

            session_file=session_entry.get(),

            start_date=start_entry.get(),

            end_date=end_entry.get()
        )

        messagebox.showinfo(

            "Success",

            "Automation completed!"
        )

    except Exception as e:

        print(e)

        messagebox.showerror(

            "Error",

            str(e)
        )

tk.Button(

    root,

    text="Run",

    bg="green",

    fg="white",

    command=run

).pack(pady=20)

root.mainloop()